# Full-range RAISE pass (FR-011) — BTN vs BB, headless GPU

Re-solves the launch **BTN-vs-BB** single-raised pot across all 17 boards **with raises enabled** (`--raise-x 3`), so every *facing-a-bet* spot gains a **Raise** option (currently those spots are Fold/Call only). First-to-act and checked-to spots stay Check/Bet — there's no bet to raise there.

The raise tree is bigger, so this runs in **3 parts** (~4–6 h each) to stay under Kaggle's ~9 h commit limit.

**Per commit:** set `PART` to `'A'`, then `'B'`, then `'C'`, and Save & Run All each time (Accelerator → GPU T4 x2, Internet On). Download `records_raise_<PART>.json` from each and send all three back — they merge into a signed full-range raise pack locally.

In [ ]:
# Fail fast if no GPU.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Accelerator -> GPU, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, 'NO GPU -- set Accelerator -> GPU T4 x2, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Clone the solver source (needs Internet On) -- pulls BTN-vs-BB + 17 boards.
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import sys; sys.path.insert(0, '/kaggle/working/poker/src')
from pokertrainer.presets import SCENARIOS, BOARDS
print('scenario:', SCENARIOS['btn_vs_bb_srp']['label'], '| boards:', len(BOARDS), '| raise-to: 3x the bet')

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

## Solve one part (~4–6 h). Set PART to 'A', then 'B', then 'C' across three commits.

Raise makes the tree bigger than the check/bet pass, so 6 boards is about one commit. If a part ever times out, split its ROOTS across two commits (e.g. `'0,1,2'` then `'3,4,5'`) — the merge is board-wise, so partial parts still combine cleanly.

In [ ]:
PART = 'A'   # <-- 'A' (boards 0-5), then 'B' (6-11), then 'C' (12-16)
ROOTS = {'A': '0,1,2,3,4,5', 'B': '6,7,8,9,10,11', 'C': '12,13,14,15,16'}[PART]
import subprocess, os
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.run(
    ['python', '-m', 'pokertrainer.content_yield',
     '--solver', 'gpu', '--dtype', 'float32', '--n', '400', '--iters', '300',
     '--scenario', 'btn_vs_bb_srp', '--raise-x', '3', '--roots', ROOTS,
     '--out', f'/kaggle/working/cy_raise_{PART}'],
    cwd='/kaggle/working/poker', env=env)

In [ ]:
# Expose records_raise_<PART>.json for download (fails loudly if nothing completed).
import json, shutil, os
try:
    rep = json.load(open(f'/kaggle/working/cy_raise_{PART}/yield_report.json'))
except (FileNotFoundError, json.JSONDecodeError):
    raise SystemExit(f'No cy_raise_{PART} output -- check the cell above and cy_raise_{PART}/boards/*.ERROR.txt')
done = rep.get('boards_completed') or []
print('PART', PART, '| boards completed:', done, '| missing:', rep.get('boards_missing'))
if not done:
    raise SystemExit(f'No boards completed in PART {PART} -- check cy_raise_{PART}/boards/*.ERROR.txt')
recs = json.load(open(f'/kaggle/working/cy_raise_{PART}/records.json'))
with_raise = sum(1 for r in recs if 'raise' in (r.get('actions') or []))
shutil.copy(f'/kaggle/working/cy_raise_{PART}/records.json', f'/kaggle/working/records_raise_{PART}.json')
print('DOWNLOAD -> records_raise_%s.json (%d KB, %d records, %d with a raise option)'
      % (PART, os.path.getsize(f'/kaggle/working/records_raise_{PART}.json')//1024, len(recs), with_raise))